    docker exec -it cluster-master bash 
    
    apt update -y
    
    apt install -y  python3-pip
    
    python3 -m pip install --upgrade pip
    
    python3 -m pip install jupyterlab findspark "pandas>=1.0.5"

In [1]:
import findspark
findspark.init("/usr/local/spark")
from pyspark.sql import SparkSession, functions as F

In [2]:
spark = SparkSession.builder \
.appName("Spark YARN Example") \
.master("yarn") \
.config("spark.jars.packages", "org.postgresql:postgresql:42.6.0") \
.enableHiveSupport() \
.getOrCreate()

:: loading settings :: url = jar:file:/usr/local/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-88097250-4609-486b-84b8-331a2e1f749c;1.0
	confs: [default]
	found org.postgresql#postgresql;42.6.0 in central
	found org.checkerframework#checker-qual;3.31.0 in central
:: resolution report :: resolve 609ms :: artifacts dl 28ms
	:: modules in use:
	org.checkerframework#checker-qual;3.31.0 from central in [default]
	org.postgresql#postgresql;42.6.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   2   |   0   |   0   |   0   ||   2   |   0   |
	--------------------------------------------------------

In [3]:
# curl -O https://raw.githubusercontent.com/erkansirin78/datasets/master/Advertising.csv
# hdfs dfs -put Advertising.csv /user/root/

In [4]:
df_hdfs = spark.read.format("csv").option("header", True) \
.option("inferSchema", True).option("sep",",") \
.load("hdfs://cluster-master:9000/user/root/Advertising.csv")

In [5]:
df_hdfs.show(5)

+---+-----+-----+---------+-----+
| ID|   TV|Radio|Newspaper|Sales|
+---+-----+-----+---------+-----+
|  1|230.1| 37.8|     69.2| 22.1|
|  2| 44.5| 39.3|     45.1| 10.4|
|  3| 17.2| 45.9|     69.3|  9.3|
|  4|151.5| 41.3|     58.5| 18.5|
|  5|180.8| 10.8|     58.4| 12.9|
+---+-----+-----+---------+-----+
only showing top 5 rows



In [6]:
spark.sql("show databases").show()

+---------+
|namespace|
+---------+
|  default|
|    test1|
+---------+



In [7]:
spark.sql("use test1")

DataFrame[]

In [8]:
spark.sql("show tables;").show()

+---------+-------------+-----------+
|namespace|    tableName|isTemporary|
+---------+-------------+-----------+
|    test1|spark_adv_orc|      false|
+---------+-------------+-----------+



In [11]:
df_hdfs.write.mode("overwrite").format("orc").saveAsTable("test1.spark_adv_orc")

In [12]:
spark.sql("show tables;").show()

+---------+-------------+-----------+
|namespace|    tableName|isTemporary|
+---------+-------------+-----------+
|    test1|spark_adv_orc|      false|
+---------+-------------+-----------+



In [13]:
df_hive = spark.table("test1.spark_adv_orc")

In [14]:
df_hive.show(4)

+---+-----+-----+---------+-----+
| ID|   TV|Radio|Newspaper|Sales|
+---+-----+-----+---------+-----+
|  1|230.1| 37.8|     69.2| 22.1|
|  2| 44.5| 39.3|     45.1| 10.4|
|  3| 17.2| 45.9|     69.3|  9.3|
|  4|151.5| 41.3|     58.5| 18.5|
+---+-----+-----+---------+-----+
only showing top 4 rows

